# Nutrify v2 — Fine-tune Llama 3.2 1B para estimación de calorías
**Ejecutar TODAS las celdas en orden. Runtime → T4 GPU.**
**Dataset: dataset-v2-final.jsonl (3252 entries)**

In [ ]:
%%capture
# Celda 1: Instalar dependencias (~2 min)
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# Celda 2: Cargar modelo base + LoRA
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Modelo cargado con LoRA!")

In [ ]:
# Celda 3: Subir dataset
from google.colab import files
import json

uploaded = files.upload()  # Subí dataset-v2-final.jsonl

# Detectar el nombre del archivo subido
filename = list(uploaded.keys())[0]
print(f"Archivo subido: {filename}")

dataset_lines = []
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            dataset_lines.append(json.loads(line))

print(f"Cargadas {len(dataset_lines)} entradas de entrenamiento")
print(f"Ejemplo: {dataset_lines[0]}")

In [ ]:
# Celda 4: Formatear dataset
from datasets import Dataset

alpaca_prompt = """### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_prompts(examples):
    texts = []
    for instruction, input_text, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(dataset_lines)
dataset = dataset.map(format_prompts, batched=True)

print(f"Dataset listo: {len(dataset)} ejemplos")

In [ ]:
# Celda 5: TRAINING (~10 min con 3252 entries)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining completo!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Tiempo: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# Celda 6: Test del modelo entrenado
FastLanguageModel.for_inference(model)

test_foods = [
    "milanesa de pollo",
    "3 milanesas de pollo",
    "dos milanesas con puré",
    "café con leche y tres medialunas",
    "3 empanadas de carne",
    "media docena de empanadas",
    "big mac con papas medianas",
    "yogur ser firme",
    "fernet con coca",
    "almorcé un guiso de lentejas",
    "2 porciones de pizza con coca",
    "tostado de jamón y queso",
    "me clavé 4 facturas",
    "alfajor havanna",
]

for food in test_foods:
    prompt = alpaca_prompt.format("Estimá las calorías de esta comida", food, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            do_sample=False,
            use_cache=False,
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = result.split("### Response:\n")[-1].strip()
    print(f"{food:45s} → {response}")

In [ ]:
# Celda 7: Limpiar exports anteriores + Mergear modelo
import shutil, os

# Limpiar para asegurar que no use cache viejo
for d in ["nutrify-merged", "outputs"]:
    if os.path.exists(d):
        shutil.rmtree(d)
for f in ["nutrify-q8.gguf"]:
    if os.path.exists(f):
        os.remove(f)

print("Cache limpiado")

# Mergear LoRA weights con modelo base
model.save_pretrained_merged(
    "nutrify-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Modelo mergeado!")

In [ ]:
# Celda 8: Instalar conversor GGUF
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q ./llama.cpp/gguf-py
print("Conversor listo!")

In [ ]:
# Celda 9: Convertir a GGUF Q8
!python llama.cpp/convert_hf_to_gguf.py nutrify-merged --outfile nutrify-q8.gguf --outtype q8_0

import os
size = os.path.getsize("nutrify-q8.gguf") / 1024 / 1024
print(f"\nnutrify-q8.gguf — {size:.1f} MB")

In [ ]:
# Celda 10: Verificar que el modelo es NUEVO (hash debe ser diferente al viejo)
import hashlib

h = hashlib.sha256()
with open("nutrify-q8.gguf", "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        h.update(chunk)

new_hash = h.hexdigest()
old_hash = "2660c4946b68972141e88019cfab849c0c982f9e9b2f345de2c1aea6fdafccdb"

print(f"SHA256: {new_hash}")
if new_hash == old_hash:
    print("⚠️  MISMO HASH QUE EL MODELO VIEJO — el LoRA no se aplicó correctamente!")
    print("No descargues este archivo, algo falló en el merge.")
else:
    print("✅ Hash diferente — modelo nuevo confirmado! Descargá el archivo.")

In [ ]:
# Celda 11: Descargar modelo
from google.colab import files
files.download("nutrify-q8.gguf")